# 外汇汇率预测 Transformer 模型

本笔记演示了如何使用 PyTorch 构建 Transformer 模型，对 Excel 中的外汇汇率特征进行建模、训练与预测，同时包含自动超参数搜索、指标评估与可视化分析。

## 使用说明
1. 在下方的 **Configuration** 区块中，根据自己的 Excel 文件路径及字段，指定因变量列 (`CONFIG['target_columns']`)；未列出的（除日期列外）其余列默认作为自变量。
2. 确保 Excel 文件中包含 `Date` 日期列，或在配置中指定正确的日期列名称。
3. 运行每个代码单元格以完成数据加载、特征工程、超参数搜索、模型训练、评估与可视化。


In [ ]:
# Configuration
from pathlib import Path

CONFIG = {
    "excel_path": Path("data/forex_features.xlsx"),  # Excel 文件路径
    "date_column": "Date",  # 日期列名称
    "target_columns": [
        # "Exchange_Rate",  # 在此列出需要预测的因变量列名
    ],
    "test_ratio": 0.2,  # 测试集比例
    "val_ratio": 0.1,  # 验证集比例（用于超参数搜索）
    "lookback": 30,  # 序列窗口长度
    "horizon": 1,  # 预测步长（1 表示预测下一期）
    "epochs": 100,
    "seed": 42,
    "device": "cuda",  # 如果有 GPU，可设置为 "cuda"，否则保持 "cpu"
    "default_hyperparameters": {
        "d_model": 64,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 128,
        "dropout": 0.1,
        "batch_size": 32,
        "learning_rate": 1e-3,
    },
    "hyperparameter_grid": {
        "d_model": [64, 96],
        "nhead": [4, 8],
        "num_layers": [2, 3],
        "dim_feedforward": [128, 256],
        "dropout": [0.1, 0.2],
        "batch_size": [32],
        "learning_rate": [1e-3, 5e-4],
    },
    "gradient_clip": 1.0,
}


In [ ]:
# Imports
import copy
import math
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import ParameterGrid
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
device = torch.device(CONFIG["device"] if torch.cuda.is_available() and CONFIG["device"] == "cuda" else "cpu")
print(f"Using device: {device}")


In [ ]:
# Data Loading
excel_path = Path(CONFIG["excel_path"])
if not excel_path.exists():
    raise FileNotFoundError(f"未找到 Excel 文件: {excel_path.resolve()}")

date_col = CONFIG["date_column"]
raw_df = pd.read_excel(excel_path, parse_dates=[date_col])
if raw_df[date_col].isna().any():
    raise ValueError("日期列包含缺失值，请清理数据后再运行。")

candidate_target_cols = CONFIG.get("target_columns")
if not candidate_target_cols:
    raise ValueError("请在 CONFIG['target_columns'] 中至少指定一个因变量列名。")

if isinstance(candidate_target_cols, str):
    candidate_target_cols = [candidate_target_cols]
else:
    candidate_target_cols = list(candidate_target_cols)

missing_targets = [col for col in candidate_target_cols if col not in raw_df.columns]
if missing_targets:
    raise ValueError(f"以下因变量列在数据中不存在: {missing_targets}")

df = raw_df.drop_duplicates(subset=[date_col]).set_index(date_col).sort_index()
target_cols = list(dict.fromkeys(candidate_target_cols))
feature_cols = [col for col in df.columns if col not in target_cols]

if not feature_cols:
    raise ValueError("除因变量外没有其他列可作为自变量，请检查数据。")

print(f"Target columns: {target_cols}")
print(f"Feature columns: {feature_cols}")

display(df.head())


In [ ]:
# Descriptive Statistics
selected_cols = list(dict.fromkeys(feature_cols + target_cols))
display(df[selected_cols].describe().T)
display(pd.DataFrame({"missing_values": df[selected_cols].isna().sum()}))


In [ ]:
# Train/Validation/Test Split and Scaling
test_ratio = CONFIG["test_ratio"]
val_ratio = CONFIG["val_ratio"]
lookback = CONFIG["lookback"]
horizon = CONFIG["horizon"]

if not 0 < test_ratio < 1:
    raise ValueError("test_ratio 必须在 0 与 1 之间。")

if not 0 < val_ratio < 1:
    raise ValueError("val_ratio 必须在 0 与 1 之间。")

if test_ratio + val_ratio >= 1:
    raise ValueError("test_ratio 与 val_ratio 之和必须小于 1。")

if lookback <= 0:
    raise ValueError("lookback 必须为正整数。")

if horizon <= 0:
    raise ValueError("horizon 必须为正整数。")

total_len = len(df)
test_size = max(int(total_len * test_ratio), horizon)
val_size = max(int(total_len * val_ratio), horizon)
train_size = total_len - val_size - test_size

if train_size <= lookback:
    raise ValueError("训练集样本过少，无法满足 lookback 要求，请调整数据或参数。")

train_df = df.iloc[:train_size].copy()
val_df = df.iloc[train_size:train_size + val_size].copy()
test_df = df.iloc[train_size + val_size :].copy()

if len(val_df) < horizon:
    raise ValueError("验证集样本过少，无法生成序列，请增加 val_ratio。")

if len(test_df) < horizon:
    raise ValueError("测试集样本过少，无法生成序列，请调整 test_ratio 或 horizon。")

feature_scaler_train = StandardScaler().fit(train_df[feature_cols])
target_scaler_train = StandardScaler().fit(train_df[target_cols])

def create_sequences(feature_array, target_array, lookback, horizon, start_offset=0):
    seq_features, seq_targets = [], []
    total_length = len(feature_array)
    upper_bound = total_length - horizon + 1
    if upper_bound <= lookback:
        return seq_features, seq_targets
    for idx in range(lookback, upper_bound):
        target_position = idx + horizon - 1
        if target_position < start_offset:
            continue
        seq_features.append(feature_array[idx - lookback : idx])
        seq_targets.append(target_array[target_position])
    return seq_features, seq_targets

class SequenceDataset(Dataset):
    def __init__(self, features, targets):
        self.features = features
        self.targets = targets

    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

def build_sequence_dataset(
    core_df,
    *,
    feature_scaler,
    target_scaler,
    lookback,
    horizon,
    feature_cols,
    target_cols,
    context_df=None,
    dataset_name="dataset",
):
    if core_df.empty:
        raise ValueError(f"{dataset_name} 数据为空，请调整划分比例。")

    frames = []
    if context_df is not None and not context_df.empty:
        frames.append(context_df.tail(lookback))
    frames.append(core_df)
    combined = pd.concat(frames)
    start_offset = combined.shape[0] - core_df.shape[0]

    feature_array = feature_scaler.transform(combined[feature_cols]).astype(np.float32)
    target_array = target_scaler.transform(combined[target_cols]).astype(np.float32)

    seq_features, seq_targets = create_sequences(
        feature_array,
        target_array,
        lookback,
        horizon,
        start_offset=start_offset,
    )

    if not seq_features:
        raise ValueError(f"{dataset_name} 数据无法生成序列，请调整 lookback/horizon 或划分比例。")

    features = np.stack(seq_features).astype(np.float32)
    targets = np.stack(seq_targets).astype(np.float32)
    return SequenceDataset(torch.from_numpy(features), torch.from_numpy(targets))

val_context = train_df.tail(lookback)
train_dataset = build_sequence_dataset(
    train_df,
    feature_scaler=feature_scaler_train,
    target_scaler=target_scaler_train,
    lookback=lookback,
    horizon=horizon,
    feature_cols=feature_cols,
    target_cols=target_cols,
    dataset_name="train",
)

val_dataset = build_sequence_dataset(
    val_df,
    feature_scaler=feature_scaler_train,
    target_scaler=target_scaler_train,
    lookback=lookback,
    horizon=horizon,
    feature_cols=feature_cols,
    target_cols=target_cols,
    context_df=val_context,
    dataset_name="validation",
)

print(f"Train rows: {len(train_df)} | Validation rows: {len(val_df)} | Test rows: {len(test_df)}")
print(f"Train sequences: {len(train_dataset)} | Validation sequences: {len(val_dataset)}")

In [ ]:
# Model Definition & Training Utilities
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


class TimeSeriesTransformer(nn.Module):
    def __init__(
        self,
        *,
        input_size,
        output_size,
        d_model,
        nhead,
        num_layers,
        dim_feedforward,
        dropout,
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.positional_encoding = PositionalEncoding(d_model=d_model, dropout=dropout)
        self.regressor = nn.Linear(d_model, output_size)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.positional_encoding(x)
        x = self.transformer_encoder(x)
        output = self.regressor(x[:, -1, :])
        return output


def instantiate_model(params):
    return TimeSeriesTransformer(
        input_size=len(feature_cols),
        output_size=len(target_cols),
        d_model=params["d_model"],
        nhead=params["nhead"],
        num_layers=params["num_layers"],
        dim_feedforward=params["dim_feedforward"],
        dropout=params["dropout"],
    ).to(device)


def run_training(model, optimizer, train_loader, val_loader, epochs, log_prefix="", gradient_clip=None):
    history = {"train_loss": [], "val_loss": []}
    gradient_clip = gradient_clip if gradient_clip is not None else CONFIG.get("gradient_clip", 1.0)
    display_step = max(1, epochs // 10)

    if val_loader is None:
        for epoch in range(1, epochs + 1):
            model.train()
            train_loss = 0.0
            for batch_x, batch_y in train_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)

                optimizer.zero_grad()
                preds = model(batch_x)
                loss = nn.functional.mse_loss(preds, batch_y)
                loss.backward()
                if gradient_clip:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
                optimizer.step()

                train_loss += loss.item() * batch_x.size(0)

            train_loss /= len(train_loader.dataset)
            history["train_loss"].append(train_loss)
            history["val_loss"].append(np.nan)

            if epoch % display_step == 0 or epoch == 1 or epoch == epochs:
                print(f"{log_prefix}Epoch {epoch:3d} | Train Loss: {train_loss:.4f}")

        best_state = copy.deepcopy(model.state_dict())
        return history, None, best_state

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            preds = model(batch_x)
            loss = nn.functional.mse_loss(preds, batch_y)
            loss.backward()
            if gradient_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
            optimizer.step()

            train_loss += loss.item() * batch_x.size(0)

        train_loss /= len(train_loader.dataset)
        history["train_loss"].append(train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                preds = model(batch_x)
                loss = nn.functional.mse_loss(preds, batch_y)
                val_loss += loss.item() * batch_x.size(0)

        val_loss /= len(val_loader.dataset)
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        if epoch % display_step == 0 or epoch == 1 or epoch == epochs:
            print(f"{log_prefix}Epoch {epoch:3d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    model.load_state_dict(best_state)
    return history, best_val_loss, best_state

In [ ]:
# Hyperparameter Search
grid_config = CONFIG.get("hyperparameter_grid", {})
default_params = CONFIG["default_hyperparameters"].copy()

if not isinstance(grid_config, dict):
    raise ValueError("hyperparameter_grid 必须是字典。")

if not grid_config:
    param_combinations = [default_params]
else:
    param_combinations = []
    for combo in ParameterGrid(grid_config):
        params = default_params.copy()
        params.update(combo)
        param_combinations.append(params)

if not param_combinations:
    raise ValueError("未生成任何超参数组合，请检查配置。")

best_params = None
best_val_loss = float("inf")
best_history = None

for trial_idx, params in enumerate(param_combinations, start=1):
    if params["d_model"] % params["nhead"] != 0:
        print(f"[Trial {trial_idx}/{len(param_combinations)}] 跳过无效组合（d_model 需被 nhead 整除）: {params}")
        continue

    model = instantiate_model(params)
    optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"])

    train_loader = DataLoader(train_dataset, batch_size=params["batch_size"], shuffle=True, drop_last=False)
    val_loader = DataLoader(val_dataset, batch_size=params["batch_size"], shuffle=False, drop_last=False)

    history, val_loss, _ = run_training(
        model,
        optimizer,
        train_loader,
        val_loader,
        CONFIG["epochs"],
        log_prefix=f"[Trial {trial_idx}/{len(param_combinations)}] ",
    )

    print(f"[Trial {trial_idx}/{len(param_combinations)}] 最佳验证损失: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_params = params
        best_history = history

if best_params is None:
    raise RuntimeError("所有超参数组合均无效，请调整 hyperparameter_grid。")

print(f"最佳超参数: {best_params} (验证集 Loss = {best_val_loss:.4f})")

In [ ]:
# Final Training with Best Hyperparameters
train_val_df = pd.concat([train_df, val_df])
feature_scaler_final = StandardScaler().fit(train_val_df[feature_cols])
target_scaler_final = StandardScaler().fit(train_val_df[target_cols])

train_val_dataset = build_sequence_dataset(
    train_val_df,
    feature_scaler=feature_scaler_final,
    target_scaler=target_scaler_final,
    lookback=lookback,
    horizon=horizon,
    feature_cols=feature_cols,
    target_cols=target_cols,
    dataset_name="train_val",
)

test_context = train_val_df.tail(lookback)
test_dataset = build_sequence_dataset(
    test_df,
    feature_scaler=feature_scaler_final,
    target_scaler=target_scaler_final,
    lookback=lookback,
    horizon=horizon,
    feature_cols=feature_cols,
    target_cols=target_cols,
    context_df=test_context,
    dataset_name="test",
)

train_val_loader = DataLoader(train_val_dataset, batch_size=best_params["batch_size"], shuffle=True, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=best_params["batch_size"], shuffle=False, drop_last=False)

final_model = instantiate_model(best_params)
final_optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params["learning_rate"])

history, _, final_state = run_training(
    final_model,
    final_optimizer,
    train_val_loader,
    val_loader=None,
    epochs=CONFIG["epochs"],
    log_prefix="[Final] ",
)

final_model.load_state_dict(final_state)
target_scaler = target_scaler_final  # 用于后续反归一化

print(f"训练样本序列（调参）: {len(train_dataset)} | 验证样本序列: {len(val_dataset)}")
print(f"最终训练序列数: {len(train_val_dataset)} | 测试序列数: {len(test_dataset)}")

In [ ]:
# Evaluation on Test Set
final_model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        preds = final_model(batch_x).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(batch_y.numpy())

preds_scaled = np.concatenate(all_preds, axis=0)
targets_scaled = np.concatenate(all_targets, axis=0)

preds = target_scaler.inverse_transform(preds_scaled)
targets = target_scaler.inverse_transform(targets_scaled)

metrics = []
for idx, col in enumerate(target_cols):
    metrics.append(
        {
            "Target": col,
            "MAE": mean_absolute_error(targets[:, idx], preds[:, idx]),
            "RMSE": root_mean_squared_error(targets[:, idx], preds[:, idx]),
            "R^2": r2_score(targets[:, idx], preds[:, idx]),
        }
    )

metrics_df = pd.DataFrame(metrics).set_index("Target")
display(metrics_df)

history_df = pd.DataFrame(history)
ax = history_df.dropna(axis=1, how="all").plot(title="Training Loss (Final Model)", figsize=(8, 4))
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
plt.show()


In [ ]:
# Prediction Plot
prediction_index = test_df.index[horizon - 1 : horizon - 1 + len(preds)]

result_columns = {}
for idx, col in enumerate(target_cols):
    result_columns[f"Actual_{col}"] = targets[:, idx]
    result_columns[f"Predicted_{col}"] = preds[:, idx]

result_df = pd.DataFrame(result_columns, index=prediction_index)

plot_cols = [f"Actual_{target_cols[0]}", f"Predicted_{target_cols[0]}"]
ax = result_df[plot_cols].plot(figsize=(12, 5), title=f"Actual vs. Predicted ({target_cols[0]})")
ax.set_xlabel("Date")
ax.set_ylabel(target_cols[0])
plt.show()

display(result_df.head())